# Static SHRED on ME4OH full-field OFAM3 SST data - changing lag length

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
import json
import random
from collections import Counter

# Third-party imports
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
import numpy as np
import pandas as pd
import torch
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.load_sst_en411_data import crop_df_to_area
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import (
    plot_compare_sst_recon,
    plot_sensors,
    plot_sst_map,
    set_area,
)
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette

## SHRED preprocessing

In [ ]:
# Experiment 3: trying 2 week sequences
exp_index = 3

In [ ]:
# Load and reformat data
lats, longs = get_lats_longs(NA_DATA_PATH)

In [ ]:
num_sensors = 5
lags = 2
test_recons_5_2 = np.load(f"Reconstructions/recons_n{num_sensors}_l{lags}.npy")
ground_truth_5_2 = np.load(f"Reconstructions/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_5_2, sensor_coords_5_2, test_dates_5_2 = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json")


test_recons_5_52 = np.load(f"Reconstructions/recons_n5_l52.npy")
ground_truth_5_52 = np.load(f"Reconstructions/truth_n5_l52.npy")

## Evaluation

In [ ]:
loc, coords, dates = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
for i in np.random.choice(range(len(test_recons_5_2)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons_5_2[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_lag_dates[i]}.csv")
    print(f"Date: {test_lag_dates[i]}")
    # plot_compare_sst_recon(recon_df, true_df, area="NA")
    plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=sensor_coords)

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
rmse = np.sqrt(MSE(y_pred=test_recons_5_2, y_true=ground_truth_5_2))
mae = MAE(y_pred=test_recons_5_2, y_true=ground_truth_5_2)
r2 = r2_score(y_pred=test_recons_5_2, y_true=ground_truth_5_2)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(ground_truth_5_2[i], test_recons_5_2[i], 
                   data_range=(max(test_recons_5_2[i].max(), ground_truth_5_2[i].max()) - min(test_recons_5_2[i].min(), ground_truth_5_2[i].min())))
                   for i in range(len(test_recons_5_2))])
print(f"Average SSI:\t{ssi:.3f}")

In [ ]:
# ssi_by_images = ssi_by_image(test_recons_5_2[0:2], test_ground_truth[0:2], lats, longs, plot_images=True)
# ssi_by_images_mean = np.mean(ssi_by_images)
# print(f"Average SSI (by images):\t{ssi_by_images_mean:.3f}")

#### Gulf-Stream region

In [ ]:
GS_test_data = mask_array_by_lat_long(test_recons_5_2, lats, longs)
GS_truth_data = mask_array_by_lat_long(ground_truth_5_2, lats, longs)
GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

In [ ]:
rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i], data_range=test_recons_5_2[i].max() -  test_recons_5_2[i].min()) for i in range(len(test_recons_5_2))])
print(f"Average SSI:\t{ssi:.3f}")

### Regional integral of SST (area weighted mean)

In [ ]:
test_sst_mean_5_2, area_test = area_weighted_mean(test_recons_5_2, lats, longs)
truth_sst_mean_5_2, area_truth = area_weighted_mean(ground_truth_5_2, lats, longs)
test_sst_mean_5_52, _ = area_weighted_mean(test_recons_5_52, lats, longs)
truth_sst_mean_5_52, area_truth = area_weighted_mean(ground_truth_5_52, lats, longs)

In [ ]:
sst_mean_df_5_2 = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED": test_sst_mean_5_2,
    "Truth": truth_sst_mean_5_2
})
# Smooth over 1 year = 52 weeks
sst_rolling_df_5_2 = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED": sst_mean_df_5_2["SHRED"].rolling(52).mean(),
    "Truth": sst_mean_df_5_2.Truth.rolling(52).mean()
})
sst_mean_df_5_2 = sst_mean_df_5_2.set_index("Date")
sst_rolling_df_5_2 = sst_rolling_df_5_2.set_index("Date")

In [ ]:
sst_mean_df_5_52 = pd.DataFrame({
    "Date": test_lag_dates_52,
    "SHRED": test_sst_mean_5_52,
    "Truth": truth_sst_mean_5_52
})
sst_rolling_df_5_52 = pd.DataFrame({
    "Date": test_lag_dates_52,
    "SHRED": sst_mean_df_5_52["SHRED"].rolling(52).mean(),
    "Truth": sst_mean_df_5_52.Truth.rolling(52).mean()
})
sst_mean_df_5_52 = sst_mean_df_5_52.set_index("Date")
sst_rolling_df_5_52 = sst_rolling_df_5_52.set_index("Date")

In [ ]:
plt.figure(figsize=(8,8))
sst_mean_df_5_2[["SHRED", "Truth"]].plot(colormap=ListedColormap(colours[:2]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(title="5 sensors, 2 week lags", bbox_to_anchor=(1,1))
plt.show()

plt.figure(figsize=(8,8))
sst_mean_df_5_52[["SHRED", "Truth"]].plot(colormap=ListedColormap(colours[2:4]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(title="5 sensors, 52 week lags", bbox_to_anchor=(1,1))
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
sst_rolling_df_5_2[["SHRED", "Truth"]].plot(colormap=ListedColormap(colours[:2]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.legend(title="5 sensors, 2 week lags", bbox_to_anchor=(1,1))
plt.show()

plt.figure(figsize=(8,8))
sst_rolling_df_5_52[["SHRED", "Truth"]].plot(colormap=ListedColormap(colours[2:4]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.legend(title="5 sensors, 52 week lags", bbox_to_anchor=(1,1))
plt.show()

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

OR could scale by area by doing lat_band/total_area????

In [ ]:
years = [int(date.year) for date in test_lag_dates]
year_labels = np.unique(years)
Counter(years)

In [ ]:
years = [int(date.year) for date in test_lag_dates_52]
year_labels = np.unique(years)
Counter(years)

In [ ]:
difference_5_2 = test_recons_5_2 - ground_truth_5_2
difference_5_52 = test_recons_5_52 - ground_truth_5_52

In [ ]:
latitude_avg_diffs_5_2 = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference_5_2])
latitude_avg_diffs_5_52 = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference_5_52])

avg_all_time_lat_avg_diffs_5_2 = np.array([np.mean([time[j] for time in latitude_avg_diffs_5_2]) for j in range(len(latitude_avg_diffs_5_2[0]))])
avg_all_time_lat_avg_diffs_5_52 = np.array([np.mean([time[j] for time in latitude_avg_diffs_5_52]) for j in range(len(latitude_avg_diffs_5_52[0]))])

In [ ]:
plt.plot(avg_all_time_lat_avg_diffs_5_2, np.unique(lats), c=colours[0], label="2 weeks")
plt.plot(avg_all_time_lat_avg_diffs_5_52, np.unique(lats), c=colours[1], label="52 weeks")
plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend(title="Sequence length")